Note: Core active learning loop architecture adapted from [Robert Monarch](https://github.com/rmunro/pytorch_active_learning) and updated for custom metric tracking over time.

In [1]:
# Project path setup
%cd /content/drive/MyDrive/active-learning-lab/01_textbook_ai
models_path = '/content/drive/MyDrive/active-learning-lab/01_textbook_ai/models/'

/content/drive/MyDrive/active-learning-lab/01_textbook_ai


This study note is to show how different sampling strategy impact the annotation efficiency, as measured by F-score and AUC. To see how it works, we will use disaster news headlines for this analysis. As being alerted by author, the dataset is highly skewed with the non-disaster news being the majority. The author have held out 1,200 lines for evaluation. Let's have a peek:

In [ ]:
!echo "There are $(wc -l ./unlabeled_data/unlabeled_data.csv | cut -d' ' -f1) headlines, among which 1,200 are annotated:"
!wc -l ./evaluation_data/*


There are 1167423 headlines, among which 1,200 are annotated:
 1098 ./evaluation_data/not_related.csv
  102 ./evaluation_data/related.csv
 1200 total


We disregard how the evaluation data is sampled for now, with the low disaster related lines, about 0.5%, we can reasonable expect low F-score and AUC. Why? From the confusion table and F-score formula below, we see them both use TP(True Positive) as their nominator, and negative cases accounts for

- Confusion matrix

|                 | Predicted positive                     | Predicted Negative          |                                                                            |
| --------------- | -------------------------------------- | --------------------------- | -------------------------------------------------------------------------- |
| Actual Positive | True Positive <br>(**TP**)             | False Negative <br>(**FN**) | **Recall** or **TPR** <br>aka **sensitivity**<br>$\frac{TP}{TP + FN}$ |
| Actual Negative | False Positive<br> (**FP**)            | True Negative <br>(**TN**)  | $FPR = \frac{FP}{FP + TN}$                                                 |
|                 | **Precision**<br> $\frac{TP}{TP + FP}$ |                             |                                                                            |
<br>
- F-Score formula

**F-Score** = $\frac{2 \times Precision \times recall}{Precision + recall}$ <br>= $\frac{2\times TP}{2\times TP + FN + FP}$

In [ ]:
# import Python libraries
import os
import glob
import pandas as pd
from datetime import datetime
from IPython.display import display, Markdown


In [ ]:
# To verify our expection, we will use rondom sample method to annotate 400 headlines.
!active_learning.py --uncertain

In [ ]:
# parser function to retrieve model training time, f-score, AUC, and sample size from trained model name
def get_models_info(models_path=models_path):
    """Parse all model files and return DataFrame with info"""

    model_files = glob.glob(f"{models_path}*.params")
    models_data = []

    for model_path in sorted(model_files):
        filename = os.path.basename(model_path)
        # Remove .params and split by underscore
        parts = filename.replace('.params', '').split('_')

        if len(parts) >= 5:
            try:
                date_str = f"{parts[0]}_{parts[1]}"
                timestamp = datetime.strptime(date_str, '%Y%m%d_%H%M%S')

                model_info = {
                    'filename': filename,
                    'full_path': model_path,
                    'date': timestamp,
                    'fscore': float(parts[2]),
                    'auc': float(parts[3]),
                    'training_size': int(parts[4]),
                    'order': len(models_data) + 1
                }
                models_data.append(model_info)
            except (ValueError, IndexError) as e:
                print(f"Warning: Could not parse {filename}: {e}")

    return pd.DataFrame(models_data)


# Get model by index
def get_model_by_index(index, models_path=models_path):
    """Get model info by order (0 = oldest, -1 = newest)"""
    models_df = get_models_info(models_path)

    if models_df.empty:
        return None

    if index == -1:
        return models_df.iloc[index]  # Latest
    elif index >= 0:
        return models_df.iloc[index] if index < len(models_df) else None
    else:
        return models_df.iloc[index] if abs(index) <= len(models_df) else None




In [ ]:
# format text for notebook output
def format_model_result(model_info):
    """Create formatted text for Jupyter markdown"""

    if model_info is None:
        return "No model found"

    iteration_description = model_info['order']

    text = f"""
### Iteration: {iteration_description}

**Model File:** `{model_info['filename']}`

**Training Results:**
- **F-Score:** `{model_info['fscore']:.4f}`
- **AUC:** `{model_info['auc']:.4f}`
- **Training Size:** `{model_info['training_size']}` samples
- **Date/Time:** `{model_info['date'].strftime('%Y-%m-%d %H:%M:%S')}`

**Interpretation:**
- F-Score = {model_info['fscore']:.4f} hints {interpret_fscore(model_info['fscore'])}
- AUC = {model_info['auc']:.4f} hints {interpret_auc(model_info['auc'])}

---
"""
    return text

def interpret_fscore(fscore):
    if fscore == 0:
        return "Model failed completely"
    elif fscore < 0.5:
        return "Model performs poorly"
    elif fscore < 0.7:
        return "Model shows some ability but needs improvement"
    elif fscore < 0.8:
        return "Model is reasonably effective"
    elif fscore < 0.9:
        return "Model performs well"
    else:
        return "Model performs excellently"

def interpret_auc(auc):
    if auc < 0.5:
        return "Model is worse than random (something is wrong)"
    elif auc < 0.6:
        return "Model barely above random guessing"
    elif auc < 0.7:
        return "Model shows weak discrimination ability"
    elif auc < 0.8:
        return "Model has acceptable discrimination"
    elif auc < 0.9:
        return "Model has good discrimination"
    else:
        return "Model has excellent discrimination"

In [ ]:
# Get models
models_df = get_models_info()
markdown_text = format_model_result(models_df.iloc[0])

# Display as formatted markdown
display(Markdown(markdown_text))


### Iteration: 1

**Model File:** `20260522_142811_0.0_0.668_400.params`

**Training Results:**
- **F-Score:** `0.0000`
- **AUC:** `0.6680`
- **Training Size:** `400` samples
- **Date/Time:** `2026-05-22 14:28:11`

**Interpretation:**
- F-Score = 0.0000 hints Model failed completely
- AUC = 0.6680 hints Model shows weak discrimination ability

---


In [ ]:
!wc -l ./training_data/*

  354 ./training_data/not_related.csv
   58 ./training_data/related.csv
  412 total
